# Benchmark faster-whisper trên Google Colab (chạy từ A đến Z)

**Chuẩn bị:** upload cả thư mục `googleColab/` (4 file `.py` + notebook này) lên Google Drive tại `MyDrive/googleColab/`. Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy các cell từ trên xuống.

Notebook sẽ: cài thư viện, tải VIVOS (chỉ phần test) từ Hugging Face, chạy lần lượt mọi cấu hình, và **chỉ lưu `summary.csv`, `predictions.csv` lên Drive** (cập nhật sau mỗi cấu hình). `raw/` và `payloads/` nằm trên đĩa tạm của Colab, mất khi ngắt kết nối.

Lưu ý khi đọc kết quả:
- Colab chỉ có ít nhân CPU và GPU chia sẻ, nên số liệu **không so sánh trực tiếp** với máy cá nhân. Thông tin phần cứng được ghi vào cột cuối của `summary.csv`.
- Mỗi worker tự đo mức nền và tính các chỉ số ròng (`cpu_net_pct_eval`, `ram_net_peak_mb`, `vram_net_peak_mb`...). Trên Colab `gpu_proc_mb` có thể đọc được (Linux), khi đó `vram_net_source = process`.
- Bản Colab miễn phí có thể tự ngắt khi rảnh hoặc sau vài giờ. Nếu bị ngắt, `summary.csv` trên Drive giữ kết quả các cấu hình đã xong, nhưng chạy lại sẽ bắt đầu từ đầu vì `raw/` đã mất.

In [ ]:
# ===== Cấu hình (sửa ở đây) =====
N_EVAL = 200            # 200 = manifest_eval.csv (seed 42, trùng máy cục bộ); None = cả 760 câu test
REPEATS = 3            # đặt 3 để có độ lệch chuẩn (lâu gấp 3)
COOLDOWN_S = 5          # nghỉ giữa các lần chạy
TIMEOUT_S = 60 * 60     # quá hạn thì kill worker
RUN_LONG_AUDIO = False
RUN_CPU_LARGE = True   # True: thêm medium/large-v3 trên CPU (Colab ~2 nhân nên rất lâu)

CODE_DIR = "/content/drive/MyDrive/googleColab"                    # nơi bạn upload 4 file .py
OUT_DIR = "/content/drive/MyDrive/faster_whisper_bench_results"    # nơi lưu summary.csv, predictions.csv
WORK = "/content/work"                                             # đĩa tạm: code, dữ liệu, raw, payloads

CONFIGS = []
for m in ("tiny", "base", "small"):
    CONFIGS += [(m, "cpu", "int8")]
for m in ("tiny", "base", "small", "medium", "large-v3"):
    CONFIGS += [(m, "cuda", ct) for ct in ("float16", "int8")]
if RUN_CPU_LARGE:
    CONFIGS += [("medium", "cpu", "int8"), ("large-v3", "cpu", "int8")]
print(len(CONFIGS), "cấu hình x", REPEATS, "lần")

15 cấu hình x 3 lần


In [ ]:
# ===== Mount Drive, chép code về đĩa tạm, cài thư viện =====
import os, shutil, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

PY_FILES = ["bench_common.py", "bench_worker.py", "bench_metrics.py", "colab_data.py"]
missing = [f for f in PY_FILES if not os.path.isfile(f"{CODE_DIR}/{f}")]
if missing:                                    # không thấy trên Drive -> cho upload trực tiếp
    print("Thiếu trên Drive:", missing, "-> chọn các file .py để upload")
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        open(f"{WORK}/{name}", "wb").write(data)
else:
    for f in PY_FILES:
        shutil.copy(f"{CODE_DIR}/{f}", f"{WORK}/{f}")
assert all(os.path.isfile(f"{WORK}/{f}") for f in PY_FILES), "Vẫn thiếu file .py"

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip install -q faster-whisper==1.2.1 "av>=14,<19" jiwer psutil nvidia-ml-py nvidia-cublas-cu12

import ctranslate2
print("CUDA devices:", ctranslate2.get_cuda_device_count(), "| ctranslate2", ctranslate2.__version__)
print("cuda compute types:", sorted(ctranslate2.get_supported_compute_types("cuda")) if ctranslate2.get_cuda_device_count() else "không có GPU")
assert ctranslate2.get_cuda_device_count() > 0, "Chưa bật GPU: Runtime -> Change runtime type -> T4 GPU"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thiếu trên Drive: ['bench_common.py', 'bench_worker.py', 'bench_metrics.py', 'colab_data.py'] -> chọn các file .py để upload


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
CUDA devices: 1 | ctranslate2 4.8.3
cuda compute types: ['float16', 'float32', 'int8', 'int8_float16', 'int8_float32']


In [ ]:
# ===== Tải VIVOS test và tạo manifest =====
os.chdir(WORK)
sys.path.insert(0, WORK)
import colab_data
colab_data.download_and_extract_test(f"{WORK}/data")
full_manifest, eval_manifest = colab_data.build_manifests(f"{WORK}/data")
os.environ["BENCH_MANIFEST"] = str(eval_manifest if N_EVAL == 200 else full_manifest)   # đặt TRƯỚC khi import bench_common
print("Dùng manifest:", os.environ["BENCH_MANIFEST"])

Đã có dữ liệu test, bỏ qua tải.
manifest.csv: 760 câu | manifest_eval.csv: 200 câu
Dùng manifest: /content/work/data/manifest_eval.csv


In [ ]:
# ===== Chạy toàn bộ cấu hình; sau mỗi cấu hình xuất summary/predictions lên Drive =====
from bench_common import *
import shutil, subprocess, psutil

def hardware_info():
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    return {"hw_gpu": gpu, "hw_cpu_count": psutil.cpu_count(),
            "hw_ram_total_gb": round(psutil.virtual_memory().total / 2**30, 1),
            "n_eval_files": len(pd.read_csv(MANIFEST_PATH)), "platform": "Google Colab"}

def export_to_drive(df):
    summary = summarize(df)
    if summary.empty:
        return
    for k, v in hardware_info().items():
        summary[k] = v
    summary.to_csv(f"{OUT_DIR}/summary.csv", index=False)
    export_predictions(df)                                   # ghi predictions.csv vào RESULTS_DIR (đĩa tạm)
    shutil.copy(RESULTS_DIR / "predictions.csv", f"{OUT_DIR}/predictions.csv")
    print(f"  đã cập nhật summary.csv, predictions.csv trên Drive ({len(summary)} cấu hình)", flush=True)

df = run_all(CONFIGS, REPEATS, None, RUN_LONG_AUDIO, COOLDOWN_S, TIMEOUT_S, on_config_done=export_to_drive)
show_status(df)

15 cấu hình x 3 lần | 200 file
Kiểm tra/tải model 'base' ...
Kiểm tra/tải model 'large-v3' ...
Kiểm tra/tải model 'medium' ...
Kiểm tra/tải model 'small' ...
Kiểm tra/tải model 'tiny' ...

=== [1/15] tiny | cpu | int8 | lần 1/3 | tiến độ 0% (0/45) | đã chạy 0.0 phút
  đang nạp model ...
  nạp xong (0.7s), đang khởi động ...
  đang đánh giá: 10/200 file (5%)
  đang đánh giá: 20/200 file (10%)
  đang đánh giá: 30/200 file (15%)
  đang đánh giá: 40/200 file (20%)
  đang đánh giá: 50/200 file (25%)
  đang đánh giá: 60/200 file (30%)
  đang đánh giá: 70/200 file (35%)
  đang đánh giá: 80/200 file (40%)
  đang đánh giá: 90/200 file (45%)
  đang đánh giá: 100/200 file (50%)
  đang đánh giá: 110/200 file (55%)
  đang đánh giá: 120/200 file (60%)
  đang đánh giá: 130/200 file (65%)
  đang đánh giá: 140/200 file (70%)
  đang đánh giá: 150/200 file (75%)
  đang đánh giá: 160/200 file (80%)
  đang đánh giá: 170/200 file (85%)
  đang đánh giá: 180/200 file (90%)
  đang đánh giá: 190/200 file (95%)


In [ ]:
# ===== Bảng tổng hợp và biểu đồ =====
pd.set_option("display.max_columns", None); pd.set_option("display.width", 250)
summary = pd.read_csv(f"{OUT_DIR}/summary.csv")
display(summary)
plot_wer_vs_rtf(summarize(load_results()))
print("File trên Drive:", os.listdir(OUT_DIR))